# World Cup 2026 Bluesky Firehose — Producer
**Team:** Bojan Ivanovski, Andrada Demian

This notebook is the **producer** half of a Spark-Streaming hackathon project.

It connects to the public [Bluesky Jetstream](https://docs.bsky.app/blog/jetstream)
WebSocket, filters posts that mention the FIFA World Cup 2026 / football using a
strict word-boundary regex, parses each event into a stable JSON schema, and
broadcasts the matching posts to any consumer connected to TCP port 9999.

The consumer half is in `consumer.ipynb`. See `README.md` for setup and run
instructions.

In [1]:
%pip install websockets pyspark

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 KB 3.8 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.4/455.4 MB 5.6 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.0/203.0 KB 8.4 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-4.1.2-py2.py3-none-any.whl size=456079474 sha256=77cc8f8bdfa980345173648945ad9fc1b092a0f0d577e65c100e2f6ee352c75d
  Stored in directory: /root/.cache/pip/wheels/86/69/e0/27cb5969d983c3f3a9d3f0fc2cd85694920c9885c45659bdaa
Successfully built pyspark
Note: you may need to restart the kernel to use updated packages.


In [2]:
#Imports & Config
import asyncio
import websockets
import json
import socket
import threading
import re
from datetime import datetime, timezone

# --- Configuration ---
BLUESKY_URI = "wss://jetstream2.us-east.bsky.network/subscribe?wantedCollections=app.bsky.feed.post"

# Strict football / World Cup filter. Word-boundary regex avoids matching
# "2026" inside CVE IDs, dates, or "friendly reminder" boilerplate.
KEYWORD_PATTERNS = [
    r"\bworld\s*cup\b",
    r"\bworldcup\b",
    r"\bfifa\b",
    r"\bwcq\b",
    r"\bfootball\b",
    r"\bsoccer\b",
    r"\bmessi\b",
    r"\bmbapp[eé]\b",
    r"\bronaldo\b",
    r"\bneymar\b",
    r"\bhaaland\b",
    r"#worldcup2026",
    r"#fifaworldcup",
    r"#fifa\b",
    r"\busa2026\b",
    r"\bcanada2026\b",
    r"\bmexico2026\b",
]
TOPIC_RE = re.compile("|".join(KEYWORD_PATTERNS), re.IGNORECASE)

# TCP socket server settings (consumer will connect here)
TCP_HOST = "0.0.0.0"
TCP_PORT = 9999

In [3]:
#JSON Schema Parser
# Extracts a clean, agreed-upon JSON record from a raw Bluesky event.

def parse_event(raw_json: str) -> dict | None:

    try:
        event = json.loads(raw_json)
    except json.JSONDecodeError:
        return None

    # Only process post creation events
    if event.get("kind") != "commit":
        return None
    commit = event.get("commit", {})
    if commit.get("collection") != "app.bsky.feed.post":
        return None
    if commit.get("operation") not in ("create",):
        return None

    record = commit.get("record", {})
    text = record.get("text", "")

    # --- Topic filter: keep only World Cup related posts ---
    if not TOPIC_RE.search(text):
        return None

    # --- Extract creation timestamp ---
    created_at = record.get("createdAt", datetime.now(timezone.utc).isoformat())

    # --- Extract reply_to (reference post) ---
    reply_to = None
    reply = record.get("reply")
    if reply:
        parent = reply.get("parent", {})
        reply_to = parent.get("uri")  # e.g. at://did:.../app.bsky.feed.post/...

    # --- Extract mentions (@handles embedded as facets) ---
    mentions = []
    urls = []
    for facet in record.get("facets", []):
        for feature in facet.get("features", []):
            ftype = feature.get("$type", "")
            if ftype == "app.bsky.richtext.facet#mention":
                mentions.append(feature.get("did", ""))
            elif ftype == "app.bsky.richtext.facet#link":
                urls.append(feature.get("uri", ""))

    # --- Extract hashtags via regex (not always in facets) ---
    hashtags = re.findall(r"#(\w+)", text)

    # --- Build the agreed schema ---
    structured = {
        "did": event.get("did"),                        # author DID
        "post_uri": f"at://{event.get('did')}/app.bsky.feed.post/{commit.get('rkey')}",
        "text": text,
        "created_at": created_at,                       # ISO 8601
        "ingested_at": datetime.now(timezone.utc).isoformat(),
        "reply_to": reply_to,                           # URI of parent post or None
        "mentions": mentions,                           # list of mentioned DIDs
        "urls": urls,                                   # list of external URLs
        "hashtags": hashtags,                           # list of hashtag strings
        "lang": record.get("langs", [None])[0],        # first declared language
    }
    return structured

In [4]:
# TCP Socket Server
# Runs in a background thread. Keeps a list of connected consumer clients
# and broadcasts each post line to all of them.

clients = []
clients_lock = threading.Lock()

def start_tcp_server():
    """Start a TCP server that accepts consumer connections."""
    server_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server_sock.bind((TCP_HOST, TCP_PORT))
    server_sock.listen(5)
    print(f"[TCP] Listening on {TCP_HOST}:{TCP_PORT}")

    while True:
        conn, addr = server_sock.accept()
        print(f"[TCP] Consumer connected from {addr}")
        with clients_lock:
            clients.append(conn)

def broadcast(line: str):
    """Send a newline-delimited JSON line to all connected consumers."""
    dead = []
    with clients_lock:
        for conn in clients:
            try:
                conn.sendall((line + "\n").encode("utf-8"))
            except (BrokenPipeError, OSError):
                dead.append(conn)
        for conn in dead:
            clients.remove(conn)
            print("[TCP] Dropped a dead consumer connection.")

# Start the TCP server in a daemon thread so it doesn't block the notebook
tcp_thread = threading.Thread(target=start_tcp_server, daemon=True)
tcp_thread.start()
print("[Producer] TCP server thread started.")

[TCP] Listening on 0.0.0.0:9999
[Producer] TCP server thread started.


In [5]:
# Bluesky Jetstream Listener
# Connects to Bluesky, filters events, and broadcasts matching posts.

RECONNECT_DELAY = 5   # seconds to wait before reconnecting after a drop

async def listen_and_produce():
    """
    Main producer loop:
    1. Connect to Bluesky Jetstream
    2. Parse & filter every event
    3. Broadcast matching posts to consumers over TCP
    Automatically reconnects on disconnection.
    """
    post_count = 0
    match_count = 0

    while True:  # outer loop handles reconnects
        try:
            print(f"[Bluesky] Connecting to Jetstream...")
            async with websockets.connect(
                BLUESKY_URI,
                ping_interval=20,
                ping_timeout=20,
                max_size=2**20,  # 1 MB max message size
            ) as ws:
                print("[Bluesky] Connected. Streaming posts...")
                async for raw_message in ws:
                    post_count += 1
                    record = parse_event(raw_message)
                    if record:
                        match_count += 1
                        line = json.dumps(record, ensure_ascii=False)
                        broadcast(line)
                        # Progress feedback every 10 matches
                        if match_count % 10 == 0:
                            print(f"[Producer] {match_count} matches / {post_count} total seen | latest: {record['text'][:60]!r}")

        except websockets.ConnectionClosed as e:
            print(f"[Bluesky] Connection closed: {e}. Reconnecting in {RECONNECT_DELAY}s...")
        except Exception as e:
            print(f"[Bluesky] Unexpected error: {e}. Reconnecting in {RECONNECT_DELAY}s...")

        await asyncio.sleep(RECONNECT_DELAY)

await listen_and_produce()

[Bluesky] Connecting to Jetstream...
[Bluesky] Connected. Streaming posts...
[Producer] 10 matches / 200 total seen | latest: 'Jugará el Mundial 2026 pero en el Real Zaragoza fue un fiasc'
[Producer] 20 matches / 514 total seen | latest: 'Why has abuse of flotilla activists drawn more condemnation '
[Producer] 30 matches / 1180 total seen | latest: "Le regard de notre caricaturiste Godin sur l'actualité du 2 "
[TCP] Consumer connected from ('127.0.0.1', 46968)
[Producer] 40 matches / 1455 total seen | latest: '🏈 LAR\n\nAndrew Berry on Myles Garrett: Nothing is final at th'
[Producer] 50 matches / 1749 total seen | latest: 'Friendly reminder that you can keep track of all my game pro'
[Producer] 60 matches / 2009 total seen | latest: 'Wordle 1,809 4/6*\n\n🟨⬜🟨⬜⬜\n⬜🟩⬜🟨⬜\n🟩🟩🟩🟩⬜\n🟩🟩🟩🟩🟩\n\nYeah fair\n\nPips '
[Producer] 70 matches / 2163 total seen | latest: 'maldita.es/desinfo/2026...'
[Producer] 80 matches / 2394 total seen | latest: 'WLAN Antennas Are Annoying https://lars.ingebrigtsen.no

CancelledError: 